# Log e Registro do modelo via MLFlow Model Serving

## Pipeline Completo com Pré-processamento Integrado

Vamos criar um pipeline sklearn que inclui:
1. **Carregamento de hiperparâmetros** do modelo otimizado (`best_random_forest.joblib`)
2. **Remoção de colunas de metadados** (`id`, `dataset`, `num`) - **ANTES** do pipeline
3. **Encoding categórico** via transformer customizado
4. **Feature engineering** (age_squared, bp_chol_ratio, etc.)
5. **Normalização** (StandardScaler)
6. **Modelo** (Random Forest com hiperparâmetros otimizados)

⚠️ **Importante**: Colunas de metadados são removidas **antes** do pipeline, garantindo que sempre processamos dados limpos.

In [0]:
import warnings
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from mlflow.models.signature import infer_signature

warnings.filterwarnings('ignore')

# ============================================================================
# 1. TRANSFORMERS CUSTOMIZADOS PARA PRÉ-PROCESSAMENTO
# ============================================================================

class CategoricalEncoder(BaseEstimator, TransformerMixin):
    """Aplica encoding categórico e one-hot encoding"""
    def __init__(self):
        self.categorical_mappings = {
            'sex': {0: 'Female', 1: 'Male'},
            'cp': {0: 'typical angina', 1: 'atypical angina', 2: 'non-anginal', 3: 'asymptomatic'},
            'restecg': {0: 'normal', 1: 'st-t abnormality', 2: 'left ventricular hypertrophy'},
            'slope': {0: 'upsloping', 1: 'flat', 2: 'downsloping'},
            'thal': {0: 'normal', 1: 'fixed defect', 2: 'reversable defect'}
        }
        self.categorical_cols = ['sex', 'cp', 'restecg', 'slope', 'thal']
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        
        # Aplicar mapeamentos categóricos
        for col, mapping in self.categorical_mappings.items():
            if col in X_copy.columns:
                X_copy[col] = X_copy[col].map(lambda x: mapping.get(int(x) if pd.api.types.is_numeric_dtype(X_copy[col]) else x, x))
        
        # One-hot encoding
        cols_to_encode = [c for c in self.categorical_cols if c in X_copy.columns]
        if cols_to_encode:
            X_copy = pd.get_dummies(X_copy, columns=cols_to_encode, drop_first=True)
        
        return X_copy


class FeatureEngineer(BaseEstimator, TransformerMixin):
    """Cria features derivadas (age_squared, bp_chol_ratio, etc.)"""
    def __init__(self):
        self.eps = 1
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        
        # Age features
        if 'age' in X_copy.columns:
            X_copy['age_squared'] = X_copy['age'] ** 2
            X_copy['age_decade'] = (X_copy['age'] // 10).astype(int)
        
        # Cholesterol to age ratio
        if 'chol' in X_copy.columns and 'age' in X_copy.columns:
            X_copy['cholesterol_to_age'] = X_copy['chol'] / (X_copy['age'] + self.eps)
        
        # Max heart rate percentage
        if 'thalch' in X_copy.columns and 'age' in X_copy.columns:
            predicted_max_hr = (220 - X_copy['age']).clip(lower=1)
            X_copy['max_hr_pct'] = X_copy['thalch'] / (predicted_max_hr + self.eps)
        
        # Blood pressure to cholesterol ratio
        if 'trestbps' in X_copy.columns and 'chol' in X_copy.columns:
            X_copy['bp_chol_ratio'] = X_copy['trestbps'] / (X_copy['chol'] + self.eps)
        
        # Binary flags
        if 'fbs' in X_copy.columns:
            X_copy['fbs_flag'] = X_copy['fbs'].astype(int)
        if 'exang' in X_copy.columns:
            X_copy['exang_flag'] = X_copy['exang'].astype(int)
        
        # Stress index
        if 'thalch' in X_copy.columns and 'trestbps' in X_copy.columns:
            X_copy['stress_index'] = X_copy['thalch'] / (X_copy['trestbps'] + self.eps)
        
        # Risk interaction
        if 'age' in X_copy.columns and 'oldpeak' in X_copy.columns:
            X_copy['risk_interaction'] = X_copy['age'] * X_copy['oldpeak']
        
        # High ST depression flag
        if 'oldpeak' in X_copy.columns:
            X_copy['high_st_depression_flag'] = (X_copy['oldpeak'] > 1.0).astype(int)
        
        return X_copy


# ============================================================================
# 2. CARREGAR E PREPARAR DADOS
# ============================================================================

# Carregar dataset raw (original)
raw_data_path = "../../data/heart_disease_uci.csv"
df_raw = pd.read_csv(raw_data_path)

print(f"Dataset shape antes da limpeza: {df_raw.shape}")
print(f"Colunas: {df_raw.columns.tolist()}")

# Preparar target (num -> target binário)
if 'num' in df_raw.columns:
    df_raw['target'] = (df_raw['num'] > 0).astype(int)

# REMOVER COLUNAS DE METADADOS (ANTES DO PIPELINE)
columns_to_drop = ['id', 'dataset', 'num']
cols_dropped = [col for col in columns_to_drop if col in df_raw.columns]
if cols_dropped:
    df_raw = df_raw.drop(columns=cols_dropped)
    print(f"\nColunas removidas: {cols_dropped}")

print(f"Dataset shape após limpeza: {df_raw.shape}")
print(f"Target distribution:\n{df_raw['target'].value_counts()}")

# Separar features e target
X = df_raw.drop(columns=['target'], errors='ignore')
y = df_raw['target']

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")
X_train.head()

## Carregar Hiperparâmetros Otimizados

Vamos extrair os hiperparâmetros do modelo `best_random_forest.joblib` que foram otimizados via `ParameterSampler` na Aula 3:
- `n_estimators`: randint(50, 151)
- `max_depth`: [None, 2, 5, 7, 10, 15, 25]
- `max_features`: [2, 5, 7, 10, 15, 25]
- `max_leaf_nodes`: [None] + list(range(5, 16))
- `min_samples_split`: randint(2, 300)
- `min_samples_leaf`: randint(1, 300)

In [ ]:
# Carregar o modelo treinado para extrair hiperparâmetros
best_model_path = "../../models/best_random_forest.joblib"
trained_model = joblib.load(best_model_path)

print(f"Modelo carregado: {best_model_path}")
print(f"Tipo do modelo: {type(trained_model).__name__}")

# Verificar se é um pipeline ou modelo direto
if hasattr(trained_model, 'named_steps'):
    print("\n✓ O modelo é um Pipeline")
    print(f"Steps do pipeline: {list(trained_model.named_steps.keys())}")
    
    # Extrair o classificador do pipeline
    if 'model' in trained_model.named_steps:
        classifier = trained_model.named_steps['model']
    elif 'classifier' in trained_model.named_steps:
        classifier = trained_model.named_steps['classifier']
    else:
        # Pegar o último step
        classifier = trained_model.steps[-1][1]
else:
    print("\n✓ O modelo é um classificador direto")
    classifier = trained_model

print(f"\nClassificador: {type(classifier).__name__}")

# Extrair APENAS os hiperparâmetros otimizados no param_distributions da Aula 3
# Baseado em: param_distributions = {
#     "model__n_estimators": randint(50, 151),
#     "model__max_depth": [None, 2, 5, 7, 10, 15, 25],
#     "model__max_features": ...,
#     "model__max_leaf_nodes": [None] + list(range(5, 16)),
#     "model__min_samples_split": randint(2, 300),
#     "model__min_samples_leaf": randint(1, 300),
# }

rf_params = classifier.get_params()
relevant_params = {
    'n_estimators': rf_params.get('n_estimators'),
    'max_depth': rf_params.get('max_depth'),
    'max_features': rf_params.get('max_features'),
    'max_leaf_nodes': rf_params.get('max_leaf_nodes'),
    'min_samples_split': rf_params.get('min_samples_split'),
    'min_samples_leaf': rf_params.get('min_samples_leaf'),
    'random_state': 42  # Fixo para reprodutibilidade
}

print("\n📊 Hiperparâmetros otimizados extraídos:")
for param, value in relevant_params.items():
    print(f"  {param}: {value}")

## Construir e Treinar Pipeline Completo

In [ ]:
# ============================================================================
# 3. CRIAR PIPELINE COMPLETO COM HIPERPARÂMETROS OTIMIZADOS
# ============================================================================

# Construir pipeline usando APENAS os hiperparâmetros otimizados na Aula 3
pipeline = Pipeline([
    ('categorical_encoding', CategoricalEncoder()),
    ('feature_engineering', FeatureEngineer()),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(**relevant_params))
])

print("Pipeline criado com hiperparâmetros otimizados:")
print("="*60)
for param, value in relevant_params.items():
    print(f"  {param:25} = {value}")
print("="*60)

# Treinar pipeline
print("\nTreinando pipeline completo...")
pipeline.fit(X_train, y_train)

# Avaliar
train_score = pipeline.score(X_train, y_train)
test_score = pipeline.score(X_test, y_test)

print(f"\nTrain accuracy: {train_score:.4f}")
print(f"Test accuracy: {test_score:.4f}")

# Testar predição com uma amostra
sample = X_test.iloc[[0]]
prediction = pipeline.predict(sample)
print(f"\nSample prediction: {prediction[0]}")
print(f"Actual value: {y_test.iloc[0]}")

## Log do Pipeline Completo no MLflow

Agora vamos logar o pipeline inteiro no MLflow. O modelo registrado incluirá automaticamente todo o pré-processamento.

In [0]:
# ============================================================================
# 4. LOGAR PIPELINE NO MLFLOW
# ============================================================================

mlflow.set_experiment('/heart-disease-prediction-pipeline')

with mlflow.start_run(run_name="complete_pipeline_with_best_params"):
    # Logar hiperparâmetros do modelo
    for param_name, param_value in relevant_params.items():
        mlflow.log_param(param_name, param_value)
    
    # Logar métricas
    mlflow.log_metric("train_accuracy", train_score)
    mlflow.log_metric("test_accuracy", test_score)
    
    # Criar signature com dados limpos (entrada do pipeline)
    signature = infer_signature(X_train, pipeline.predict(X_train))
    
    # Logar o pipeline completo
    model_info = mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        signature=signature,
        input_example=X_train.iloc[:5]
    )
    
    print(f"Pipeline logged successfully!")
    print(f"Model URI: {model_info.model_uri}")
    print(f"\nHiperparâmetros logados:")
    for param_name, param_value in relevant_params.items():
        print(f"  {param_name}: {param_value}")
    
mlflow.end_run()

### 📊 Comparação com Modelo Original

Vamos comparar o desempenho do pipeline recriado com o modelo original.

In [ ]:
# Comparar com o modelo original (se for um pipeline)
if hasattr(trained_model, 'named_steps'):
    # Preparar dados para o modelo original (aplicar mesmo pré-processamento)
    # O modelo original já tem seu próprio pipeline de pré-processamento
    original_train_score = trained_model.score(X_train, y_train)
    original_test_score = trained_model.score(X_test, y_test)
    
    print("Comparação de Performance:")
    print("="*60)
    print(f"{'Modelo':<30} {'Train Acc':<15} {'Test Acc':<15}")
    print("-"*60)
    print(f"{'Pipeline Original':<30} {original_train_score:<15.4f} {original_test_score:<15.4f}")
    print(f"{'Pipeline Recriado':<30} {train_score:<15.4f} {test_score:<15.4f}")
    print("-"*60)
    
    diff_train = train_score - original_train_score
    diff_test = test_score - original_test_score
    
    print(f"{'Diferença':<30} {diff_train:+.4f} {'':>7} {diff_test:+.4f}")
    
    if abs(diff_test) < 0.01:
        print("\n✅ Performance similar ao modelo original!")
    else:
        print(f"\n⚠️ Diferença de {abs(diff_test):.2%} no test set")
else:
    print("Modelo original não é um pipeline - comparação não disponível")

## Registro do Modelo no Model Registry

In [0]:
from mlflow import MlflowClient

# Configurar Unity Catalog (ou registry local)
mlflow.set_registry_uri("databricks-uc")

# Registrar modelo
model_registered = mlflow.register_model(
    model_uri=model_info.model_uri,
    name='heart-disease-pipeline-model'
)

print(f"Model registered: {model_registered.name}")
print(f"Version: {model_registered.version}")

# Criar alias Production
client = MlflowClient()
client.set_registered_model_alias(
    model_registered.name,
    "Production",
    model_registered.version
)

print(f"Alias 'Production' set to version {model_registered.version}")

## Teste de Inferência com Dados RAW

A grande vantagem do pipeline: podemos enviar **dados raw diretamente** ao modelo!
O pipeline aplica automaticamente todo o pré-processamento.

### 💡 Decisão de Design: Remoção de Metadados

**Por que remover colunas ANTES do pipeline e não dentro dele?**

✅ **Separação de responsabilidades**: 
- Limpeza de dados = responsabilidade da aplicação/ETL
- Transformação de features = responsabilidade do pipeline ML

✅ **Flexibilidade**: 
- Diferentes fontes de dados podem ter diferentes metadados
- O pipeline foca apenas nas features relevantes

✅ **Clareza**: 
- Fica explícito que essas colunas não são features
- Evita confusão sobre quais colunas o modelo "vê"

✅ **Consistência**: 
- Garante que sempre removemos essas colunas antes de qualquer processamento
- Evita erros de esquecimento

In [0]:
# ============================================================================
# 5. TESTE LOCAL DO PIPELINE
# ============================================================================

# Carregar dados raw para teste
test_raw = pd.read_csv("../../data/heart_disease_uci.csv")

# REMOVER COLUNAS DE METADADOS (mesmo processo usado no treinamento)
columns_to_drop = ['id', 'dataset', 'num', 'target']
test_sample = test_raw.iloc[[0]].copy()
cols_to_drop = [col for col in columns_to_drop if col in test_sample.columns]
if cols_to_drop:
    test_sample = test_sample.drop(columns=cols_to_drop)

print("Dados de entrada (após remoção de metadados):")
print(test_sample.T)
print("\n" + "="*60)

# Predição com pipeline (aplica encoding + feature engineering + modelo)
prediction = pipeline.predict(test_sample)
prediction_proba = pipeline.predict_proba(test_sample)

print(f"\nPredição: {prediction[0]} (0=Sem doença, 1=Com doença)")
print(f"Probabilidades: {prediction_proba[0]}")
print(f"  - Sem doença: {prediction_proba[0][0]:.2%}")
print(f"  - Com doença: {prediction_proba[0][1]:.2%}")

In [0]:
# ============================================================================
# 6. TESTE COM MLFLOW SERVING (Databricks)
# ============================================================================

import requests
import json

TOKEN = ""  # Adicione seu token Databricks aqui

# Carregar dados RAW
raw_df = pd.read_csv('../../data/heart_disease_uci.csv')
raw_sample = raw_df.iloc[[0]].copy()

# IMPORTANTE: Remover colunas de metadados antes de enviar
columns_to_drop = ['id', 'dataset', 'num', 'target']
cols_to_drop = [col for col in columns_to_drop if col in raw_sample.columns]
if cols_to_drop:
    raw_sample = raw_sample.drop(columns=cols_to_drop)

print(f"Colunas removidas: {cols_to_drop}")
print(f"Colunas enviadas ao modelo: {raw_sample.columns.tolist()}")

# Preparar payload (dados limpos, sem metadados)
payload = {
    "dataframe_split": {
        "columns": raw_sample.columns.tolist(),
        "data": raw_sample.values.tolist()
    }
}

print("\nPayload (primeiras colunas):")
print(json.dumps({
    "dataframe_split": {
        "columns": payload["dataframe_split"]["columns"][:5],
        "data": [payload["dataframe_split"]["data"][0][:5]]
    }
}, indent=2))

# URL do endpoint (ajuste conforme necessário)
endpoint_url = "https://<workspace>.cloud.databricks.com/serving-endpoints/heart-disease-predict/invocations"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}

# Enviar requisição
if TOKEN:  # Só envia se houver token configurado
    response = requests.post(endpoint_url, headers=headers, data=json.dumps(payload))
    
    print(f"\nStatus code: {response.status_code}")
    print(f"Resposta: {response.json()}")
else:
    print("\n⚠️ Configure o TOKEN para testar o endpoint")

## Vantagens da Abordagem com Pipeline

✅ **Consistência**: O mesmo código de pré-processamento é usado em treino e inferência  
✅ **Simplicidade**: Dados raw podem ser enviados diretamente ao modelo  
✅ **Manutenibilidade**: Mudanças no pré-processamento ficam centralizadas no pipeline  
✅ **Deployment**: Pipeline completo pode ser versionado e deployado como uma unidade  
✅ **Reprodutibilidade**: Transformers customizados garantem comportamento consistente

## Carregamento e Uso do Modelo Logado

Demonstração de como carregar o pipeline do MLflow e usá-lo para predições.

In [ ]:
# ============================================================================
# 7. CARREGAR MODELO DO MLFLOW E TESTAR
# ============================================================================

# Carregar o pipeline logado do MLflow
loaded_model = mlflow.sklearn.load_model(model_info.model_uri)

print("Modelo carregado do MLflow!")
print(f"Pipeline steps: {[step[0] for step in loaded_model.steps]}")

# Testar com múltiplas amostras
test_samples = pd.read_csv("../../data/heart_disease_uci.csv").iloc[:3]

# Remover colunas de metadados
columns_to_drop = ['id', 'dataset', 'num', 'target']
cols_to_drop = [col for col in columns_to_drop if col in test_samples.columns]
if cols_to_drop:
    test_samples = test_samples.drop(columns=cols_to_drop)

predictions = loaded_model.predict(test_samples)
probabilities = loaded_model.predict_proba(test_samples)

print("\n" + "="*60)
print("PREDIÇÕES PARA 3 PACIENTES:")
print("="*60)

# Recarregar dados originais para mostrar informações
test_samples_original = pd.read_csv("../../data/heart_disease_uci.csv").iloc[:3]

for i, (pred, proba) in enumerate(zip(predictions, probabilities)):
    print(f"\nPaciente {i+1}:")
    print(f"  - Idade: {test_samples_original.iloc[i]['age']}")
    print(f"  - Sexo: {test_samples_original.iloc[i]['sex']}")
    print(f"  - Predição: {'COM doença' if pred == 1 else 'SEM doença'}")
    print(f"  - Confiança: {max(proba):.1%}")